<a href="https://colab.research.google.com/github/yshzjq/Step-2.Statistical_Thinking-Machine-Learning/blob/main/Step%202.%20%ED%86%B5%EA%B3%84%EC%A0%81%20%EC%82%AC%EA%B3%A0%EC%99%80%20%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%20%EC%9E%85%EB%AC%B8/%EC%8B%A4%EC%8A%B5_1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 데이터 표를 다루는 라이브러리 불러옴
import pandas as pd
# 숫자 계산 라이브러리 불러옴
import numpy as np
# 파일 경로 지정 (업로드된 파일 기준)
FILE_PATH = "/content/real_data_sheet1.xlsx"
# CSV 파일 읽어옴
df = pd.read_excel(FILE_PATH)
# 상위 5개 확인
print(df.head())
# 컬럼 확인
print(df.columns)

  rpt_time_date  rpt_time_time  ads_idx  mda_idx  rpt_time_clk  rpt_time_turn  \
0    2024-07-27              0     4418      761             1              0   
1    2024-07-27              0     7377      213             1              0   
2    2024-07-27              0     7377      337             1              0   
3    2024-07-27              0     7377      496             1              1   
4    2024-07-27              0     7528      792             1              0   

   rpt_time_scost  rpt_time_acost  rpt_time_cost  rpt_time_earn  
0               0               0              0              0  
1               0               0              0              0  
2               0               0              0              0  
3             230             230            180            180  
4               0               0              0              0  
Index(['rpt_time_date', 'rpt_time_time', 'ads_idx', 'mda_idx', 'rpt_time_clk',
       'rpt_time_turn', 'rpt_time_scos

In [3]:
# 고객 ID 컬럼 지정
CUSTOMER_COL = "ads_idx"   # ads_idx를 고객으로 가정함

# 사용할 숫자 컬럼들 지정 (실무에서 자주 쓰는 컬럼)
CLICK_COL = "rpt_time_clk"     # 클릭
TURN_COL  = "rpt_time_turn"    # 전환
COST_COL  = "rpt_time_cost"    # 비용(소진)
EARN_COL  = "rpt_time_earn"    # 수익

# 혹시 위에서 지정한 컬럼이 없으면 에러 나게 해서 바로 수정 가능하게 하는 기능
need_cols = [CUSTOMER_COL, CLICK_COL, TURN_COL, COST_COL, EARN_COL]

# 리스트 캄프리헨션 문법
# need_cols 중에서 df.columns에 없는 컬럼만 모아서 missing에 저장
missing = [c for c in need_cols if c not in df.columns]

if len(missing) > 0:
    raise ValueError(f"필수 컬럼이 없음: {missing} / df.columns 확인해야 함")

In [4]:
# 고객별로 클릭/전환/비용/수익을 합계하여 집계
cust = (
    df.groupby(CUSTOMER_COL)[[CLICK_COL, TURN_COL, COST_COL, EARN_COL]]
      .sum()
      .reset_index()
)
# 1. df.groupby(CUSTOMER_COL)
# -> CUSTOMER_COL(ads_idx) 기준으로 묶음(고객 단위 그룹)
# 2. [[CLICK_COL, TURN_COL, COST_COL, EARN_COL]]
# -> 그중 합산할 컬럼만 선택
# 3. .sum()
# -> 그룹별 합계 계산 (고객별 총 클릭/총 전환/총 비용/총 수익)
# 4. .reset_index()
# -> groupby 결과는 그룹키가 인덱스가 되는데 이를 다시 일반 컬럼으로 되돌림(표 형태로 보기 좋게)

In [5]:
# 파생 지표도 추가 (패턴 비교에 더 유리함)
# CVR(전환율) = 전환/클릭
cust["CVR"] = np.where(cust[CLICK_COL] > 0, cust[TURN_COL] / cust[CLICK_COL], 0)
# np.where(condition, A, B)
# -> 조건이 True면 A, False면 B

# ROAS = 수익/비용
cust["ROAS"] = np.where(cust[COST_COL] > 0, cust[EARN_COL] / cust[COST_COL], 0)

# CPA = 비용/전환
cust["CPA"] = np.where(cust[TURN_COL] > 0, cust[COST_COL] / cust[TURN_COL], 0)

# 확인
print(cust.head())
print("고객 수:", cust.shape[0]) # 행개수 = 고객수

   ads_idx  rpt_time_clk  rpt_time_turn  rpt_time_cost  rpt_time_earn  \
0     2595            36              4           2405           2405   
1     3191            82              2           1440           1800   
2     4418           297              0              0              0   
3     6508             8              0              0              0   
4     6667         19240           4391         654803         656085   

        CVR      ROAS        CPA  
0  0.111111  1.000000  601.25000  
1  0.024390  1.250000  720.00000  
2  0.000000  0.000000    0.00000  
3  0.000000  0.000000    0.00000  
4  0.228222  1.001958  149.12389  
고객 수: 15852


In [6]:
# 고객ID만 따로 저장
customer_ids = cust[CUSTOMER_COL].values
# 고객ID 컬럼을 numpy 배열로 추출
# .values: pandas Series → numpy array

# 실제 벡터로 쓸 feature 컬럼들 선택
# (합계 + 비율 지표를 섞으면 패턴 비교가 좋아짐)
feature_cols = [CLICK_COL, TURN_COL, COST_COL, EARN_COL, "CVR", "ROAS", "CPA"]

# 고객 벡터 행렬 X 만들기 (행=고객, 열=특징)
X = cust[feature_cols].values.astype(float)
# astype(float) : 숫자형으로 강제 변환(코사인 계산 안정화)

# 스케일 차이가 너무 크면(비용/수익이 압도) 유사도가 왜곡될 수 있음
# 그래서 로그 변환으로 완화
X_log = np.log1p(X)  # log(1+x) 변환함 (0도 처리 가능함)

In [7]:
# (1) 벡터 크기(노름) 계산하는 함수 만들기
def norm(v):
    # v의 길이(크기) 계산함 = sqrt(v1^2 + v2^2 + ...)
    return np.sqrt(np.sum(v * v))

# (2) 내적 계산하는 함수 만들기
def dot(a, b):
    # 내적 계산함 = a1*b1 + a2*b2 + ...
    return np.sum(a * b)

# (3) 코사인 유사도 계산하는 함수 만들기
def cosine_sim(a, b):
    # 분모가 0이면(전부 0 벡터) 계산이 안 되니까 0 처리함
    denom = norm(a) * norm(b)
    if denom == 0:
        return 0.0
    # 코사인 유사도 = 내적 / (각 벡터 크기 곱)
    return dot(a, b) / denom

In [8]:
# 비교할 기준 고객을 하나 고름 (예: 첫 번째 고객)
target_customer = customer_ids[0] # customer_ids[0] = 2595

# target 고객의 인덱스 찾음
target_idx = np.where(customer_ids == target_customer)[0][0]
# 조건을 만족하는 인덱스 배열을 반환
# -> [0][0] : 첫 번째 결과 배열에서, 첫 번째 인덱스 값을 꺼냄, (동일 ID가 여러 개면 첫 번째만 잡는 형태)
# 지금 상황에서는 어차피 target_customer id가 하나밖에 없는 상황이니 [0][0] 굳이 안해도 차이 없음

# target 고객 벡터 꺼냄 (로그 변환 버전 사용함)
target_vec = X_log[target_idx]

In [9]:
customer_ids

array([ 2595,  3191,  4418, ..., 94218, 94220, 94222])

In [10]:
np.where(customer_ids == target_customer)

(array([0]),)

In [11]:
np.where(customer_ids == target_customer)[0][0]

np.int64(0)

In [12]:
# 모든 고객과의 코사인 유사도를 계산
sims = []  # 유사도 결과 담을 리스트

for i in range(len(customer_ids)):
    # 자기 자신은 제외하고 싶으면 아래 조건 넣으면 됨
    if i == target_idx:
        continue

    # 비교 대상 고객 벡터 꺼냄
    vec_i = X_log[i]

    # 코사인 유사도 계산
    sim = cosine_sim(target_vec, vec_i)

    # (고객ID, 유사도) 저장
    sims.append((customer_ids[i], sim))

# 유사도가 큰 순서대로 정렬
sims_sorted = sorted(sims, key=lambda x: x[1], reverse=True)
# sorted(iterable, key=..., reverse=...): 정렬 함수
# key=lambda x: x[1]:
# -> 각 원소가 (고객ID, 유사도) 튜플이므로
# -> x[1](유사도) 기준으로 정렬
# reverse=True
# 내림차순(큰 유사도부터)

# TOP 5 출력
print("기준 고객:", target_customer)
print("가장 비슷한 고객 TOP 5:")

기준 고객: 2595
가장 비슷한 고객 TOP 5:


In [13]:
for cid, s in sims_sorted[:5]:
    print(cid, "-> cosine similarity:", round(s, 4))

# TOP 5 고객ID만 뽑기
top_ids = [cid for cid, _ in sims_sorted[:5]]
# TOP 5 고객ID만 리스트로 추출
# _: “두 번째 값(유사도)은 안 쓰겠다”라는 관례적 변수명

# 기준 고객 + TOP 5만 필터링
compare_df = cust[cust[CUSTOMER_COL].isin([target_customer] + top_ids)].copy()
# isin(list): CUSTOMER_COL 값이 리스트에 포함되면 True
# -> 그 True인 행만 필터링
# .copy(): 원본과 분리(경고/부작용 방지)

# 보기 좋게 정렬
compare_df = compare_df.set_index(CUSTOMER_COL).loc[[target_customer] + top_ids]
# set_index: 고객ID를 인덱스로 바꿈(보기 편하게)
# .loc[순서리스트]: 원하는 순서대로 행을 재배치 (기준 고객 → top 고객들)

# 어떤 특징이 비슷한지 표로 확인
print(compare_df[feature_cols])




28027 -> cosine similarity: 0.9999
27156 -> cosine similarity: 0.9998
26478 -> cosine similarity: 0.9998
28795 -> cosine similarity: 0.9997
47829 -> cosine similarity: 0.9997
         rpt_time_clk  rpt_time_turn  rpt_time_cost  rpt_time_earn       CVR  \
ads_idx                                                                        
2595               36              4           2405           2405  0.111111   
28027              76              5          13800          15000  0.065789   
27156              48              6           6496           6720  0.125000   
26478              87              5           8342          10428  0.057471   
28795              72              4           8550           9000  0.055556   
47829             107              7          14383          15610  0.065421   

             ROAS          CPA  
ads_idx                         
2595     1.000000   601.250000  
28027    1.086957  2760.000000  
27156    1.034483  1082.666667  
26478    1.250060  